# probit-регрессия: Качество подгонки и Сравнение моделей

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.formula.api import probit
from statsmodels.iolib.summary2 import summary_col # вывод результатов нескольких регрессий

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# подключим датасет mroz_Greene
df = pd.read_csv('../../datasets/TableF5-1.csv')
df.head()

## Спецификация и подгонка

In [ ]:
# model 1
mod_1 = probit(formula = 'LFP~WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)', data = df)
res_1 = mod_1.fit(disp=False)

In [ ]:
# model 2
mod_2 = probit(formula = 'LFP~WE+KL6+K618+CIT+UN+np.log(FAMINC)', data = df)
res_2 = mod_2.fit(disp=False)

In [ ]:
# model 3
mod_3 = probit(formula = 'LFP~WA+I(WA**2)+WE+KL6+np.log(FAMINC)', data = df)
res_3 = mod_3.fit(disp=False)

In [ ]:
# model 4
mod_4 = probit(formula = 'LFP~WA+I(WA**2)+WE+KL6', data = df)
res_4 = mod_4.fit(disp=False)

In [ ]:
# Сравнение моделей
# Имена моделей
mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']
# порядок регрессоров в таблице
reg_order = ['Intercept', 'WA', 'I(WA ** 2)', 'WE', 'KL6', 'K618', 'CIT','UN', 'np.log(FAMINC)']
# Зависимая переменная LFP
summary_col([res_1, res_2, res_3, res_4], 
            model_names=mod_names, 
            stars=True, 
            regressor_order=reg_order, 
            float_format='%.3f')

## Качество подгонки. Базовые показатели

### McFadden's $R^2$
$$ 
	R^2_{pseudo}=1-\frac{\log L_{full}}{\log L_{null}}
$$

In [ ]:
# pseudoR2 for model 1
res_1.prsquared.round(3)

In [ ]:
# pseudoR2 for model 2
res_2.prsquared.round(3)

In [ ]:
# pseudoR2 for model 3
res_3.prsquared.round(3)

In [ ]:
# pseudoR2 for model 4
res_4.prsquared.round(3)

### McFadden’s Adjusted $R^2$
$$ 
	R^2_{adj}=1-\frac{\log L_{full}-k-1}{\log L_{null}}
$$

In [ ]:
# pseudoR2.adj for model 1
(1-(res_1.llf-res_1.df_model-1)/res_1.llnull).round(3)

In [ ]:
# pseudoR2.adj for model 2
(1-(res_2.llf-res_2.df_model-1)/res_2.llnull).round(3)

In [ ]:
# pseudoR2.adj for model 3
(1-(res_3.llf-res_3.df_model-1)/res_3.llnull).round(3)

In [ ]:
# pseudoR2.adj for model 4
(1-(res_4.llf-res_4.df_model-1)/res_4.llnull).round(3)

### Cox & Snell $R^2$
$$
	R^2_{C\& S}=1-\left(\frac{L_{null}}{L_{full}}\right)^{2/n}=1-\left(\frac{\exp(\log L_{null})}{\exp(\log L_{full})}\right)^{2/n}=
	1-\exp\left(\frac{2}{n}(\log L_{null}-\log L_{full})\right)=1-\exp\left(-\frac{LR}{n}\right)
$$

In [ ]:
# Cox.Snell.R2 for model 1
(1-np.exp(-res_1.llr/res_1.nobs)).round(3)

In [ ]:
# Cox.Snell.R2 for model 2
(1-np.exp(-res_2.llr/res_2.nobs)).round(3)

In [ ]:
# Cox.Snell.R2 for model 3
(1-np.exp(-res_3.llr/res_3.nobs)).round(3)

In [ ]:
# Cox.Snell.R2 for model 4
(1-np.exp(-res_4.llr/res_4.nobs)).round(3)

### Nagelkerke / Cragg & Uhler $R^2$
$$
	R^2_{N,C\& U}=\frac{1-\left(\frac{L_{null}}{L_{full}}\right)^{2/n}}{1-L_{null}^{2/n}}=
	\frac{1-\exp\left(-\frac{LR}{n}\right)}{1-\exp(2\log L_{null}/n)}
$$

In [ ]:
# Nagelkerke.R2 for model 1
((1-np.exp(-res_1.llr/res_1.nobs))/(1-np.exp(2*res_1.llnull/res_1.nobs))).round(3)

In [ ]:
# Nagelkerke.R2 for model 2
((1-np.exp(-res_2.llr/res_2.nobs))/(1-np.exp(2*res_2.llnull/res_2.nobs))).round(3)

In [ ]:
# Nagelkerke.R2 for model 3
((1-np.exp(-res_3.llr/res_3.nobs))/(1-np.exp(2*res_3.llnull/res_3.nobs))).round(3)

In [ ]:
# Nagelkerke.R2 for model 4
((1-np.exp(-res_4.llr/res_4.nobs))/(1-np.exp(2*res_4.llnull/res_4.nobs))).round(3)

### Efron's $R^2$
$$
	R^2_{Efron}=1-\frac{\sum(y_i-\hat{P}_i)^2}{\sum(y_i-\bar{y})^2}=1-\frac{\sum(y_i-\hat{P}_i)^2}{n Var(y)}
$$

In [ ]:
# Efron.R2 for model 1
(1-(np.sum(res_1.resid_response**2))/(res_1.nobs*np.var(mod_1.endog))).round(3)

In [ ]:
# Efron.R2 for model 2
(1-(np.sum(res_2.resid_response**2))/(res_2.nobs*np.var(mod_2.endog))).round(3)

In [ ]:
# Efron.R2 for model 3
(1-(np.sum(res_3.resid_response**2))/(res_3.nobs*np.var(mod_3.endog))).round(3)

In [ ]:
# Efron.R2 for model 4
(1-(np.sum(res_4.resid_response**2))/(res_4.nobs*np.var(mod_4.endog))).round(3)

### McKelvey & Zavoina's $R^2$

\begin{align*}
	R^2_{logit}&=\frac{Var(\hat{y})}{Var(\hat{y})+\pi^2/3} & R^2_{probit}&=\frac{Var(\hat{y})}{Var(\hat{y})+1}
\end{align*}


In [ ]:
# logit
# np.var(res_1.fittedvalues)/(np.var(res_1.fittedvalues)+np.pi**2/3)

# probit
(np.var(res_1.fittedvalues)/(np.var(res_1.fittedvalues)+1)).round(3)

In [ ]:
#McKelvey.Zavoina.R2 for model 2

# probit
(np.var(res_2.fittedvalues)/(np.var(res_2.fittedvalues)+1)).round(3)

In [ ]:
#McKelvey.Zavoina.R2 for model 3

# probit
(np.var(res_3.fittedvalues)/(np.var(res_3.fittedvalues)+1)).round(3)

In [ ]:
#McKelvey.Zavoina.R2 for model 4

# probit
(np.var(res_4.fittedvalues)/(np.var(res_4.fittedvalues)+1)).round(3)